In [7]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
import os
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
import shap
import matplotlib.pyplot as plt

os.makedirs("../output", exist_ok=True)
print("All imports successful!")

All imports successful!


In [8]:
conn = sqlite3.connect("../data/attrition.db")
df = pd.read_sql("SELECT * FROM employees", conn)
conn.close()

label_cols = ["BusinessTravel", "Department", "EducationField",
              "Gender", "JobRole", "MaritalStatus", "OverTime"]
le = LabelEncoder()
df_model = df.copy()
for col in label_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

role_avg = df_model.groupby("JobRole")["MonthlyIncome"].transform("mean")
df_model["SalaryToRoleRatio"]     = (df_model["MonthlyIncome"] / role_avg).round(3)
df_model["TenureStabilityScore"]  = (df_model["YearsAtCompany"] / (df_model["TotalWorkingYears"] + 1)).round(3)
df_model["SatisfactionComposite"] = (df_model["JobSatisfaction"] + df_model["EnvironmentSatisfaction"] + df_model["WorkLifeBalance"]) / 3

drop_cols = [c for c in ["Attrition", "Attrition_Binary", "EmployeeNumber", "OverTime_Binary"] if c in df_model.columns]
X = df_model.drop(columns=drop_cols)
y = df_model["Attrition_Binary"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Data ready. Shape:", X.shape)
print("Class balance:", y.value_counts().to_dict())

Data ready. Shape: (1470, 33)
Class balance: {0: 1233, 1: 237}


In [9]:
def objective_lgbm(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 100),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1
    }
    model = LGBMClassifier(**params)
    return cross_val_score(model, X, y, cv=skf, scoring="roc_auc").mean()

print("Tuning LightGBM with Optuna (50 trials)...")
study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f"\nBest LightGBM AUC: {study_lgbm.best_value:.4f}")
print("Best params:", study_lgbm.best_params)

Tuning LightGBM with Optuna (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best LightGBM AUC: 0.8276
Best params: {'n_estimators': 120, 'max_depth': 7, 'learning_rate': 0.05822553559542543, 'num_leaves': 65, 'subsample': 0.7030716033175131, 'colsample_bytree': 0.6017801934518319, 'min_child_samples': 47}


In [10]:
def objective_gb(trial):
    params = {
        "n_estimators":   trial.suggest_int("n_estimators", 100, 300),
        "max_depth":      trial.suggest_int("max_depth", 2, 6),
        "learning_rate":  trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":      trial.suggest_float("subsample", 0.6, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "random_state": 42
    }
    model = GradientBoostingClassifier(**params)
    return cross_val_score(model, X, y, cv=skf, scoring="roc_auc").mean()

print("Tuning Gradient Boosting with Optuna (30 trials)...")
study_gb = optuna.create_study(direction="maximize")
study_gb.optimize(objective_gb, n_trials=30, show_progress_bar=True)

print(f"\nBest Gradient Boosting AUC: {study_gb.best_value:.4f}")
print("Best params:", study_gb.best_params)

Tuning Gradient Boosting with Optuna (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]


Best Gradient Boosting AUC: 0.8232
Best params: {'n_estimators': 251, 'max_depth': 2, 'learning_rate': 0.08525497143019091, 'subsample': 0.8162604756994496, 'min_samples_leaf': 16}


In [11]:
best_lgbm = LGBMClassifier(
    **study_lgbm.best_params,
    class_weight="balanced",
    random_state=42, n_jobs=-1, verbose=-1
)

best_gb = GradientBoostingClassifier(
    **study_gb.best_params,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=200, class_weight="balanced",
    random_state=42, n_jobs=-1
)

stacking_model = StackingClassifier(
    estimators=[
        ("lgbm", best_lgbm),
        ("gb",   best_gb),
        ("rf",   rf)
    ],
    final_estimator=LogisticRegression(class_weight="balanced", max_iter=1000),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1
)

print("Training stacking ensemble... takes ~3 mins")
stacking_model.fit(X, y)

y_pred_proba = stacking_model.predict_proba(X)[:, 1]
y_pred = stacking_model.predict(X)

print(f"\nStacking Ensemble AUC:      {roc_auc_score(y, y_pred_proba):.4f}")
print(f"LightGBM alone AUC:         {study_lgbm.best_value:.4f}")
print(f"Gradient Boosting alone AUC:{study_gb.best_value:.4f}")
print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=["Stayed", "Left"]))

joblib.dump(stacking_model, "../output/stacking_model.pkl")
joblib.dump(best_lgbm, "../output/lgbm_model.pkl")
print("\nModels saved.")

Training stacking ensemble... takes ~3 mins

Stacking Ensemble AUC:      0.9960
LightGBM alone AUC:         0.8276
Gradient Boosting alone AUC:0.8232

Classification Report:
              precision    recall  f1-score   support

      Stayed       1.00      0.92      0.96      1233
        Left       0.70      1.00      0.82       237

    accuracy                           0.93      1470
   macro avg       0.85      0.96      0.89      1470
weighted avg       0.95      0.93      0.93      1470


Models saved.


In [12]:
results = {}
for name, model in [("LightGBM", best_lgbm), ("GradientBoosting", best_gb), ("RandomForest", rf)]:
    auc = cross_val_score(model, X, y, cv=skf, scoring="roc_auc").mean()
    results[name] = round(auc, 4)

results["Stacking Ensemble"] = round(roc_auc_score(y, y_pred_proba), 4)

results_df = pd.DataFrame.from_dict(results, orient="index", columns=["AUC"])
results_df = results_df.sort_values("AUC", ascending=False)
print(results_df)
results_df.to_csv("../output/model_comparison.csv")

                      AUC
Stacking Ensemble  0.9960
LightGBM           0.8276
GradientBoosting   0.8232
RandomForest       0.8065


In [13]:
best_lgbm.fit(X, y)
explainer = shap.TreeExplainer(best_lgbm)
shap_values = explainer.shap_values(X)

if isinstance(shap_values, list):
    shap_left = shap_values[1]
elif len(np.array(shap_values).shape) == 3:
    shap_left = np.array(shap_values)[:, :, 1]
else:
    shap_left = shap_values

print("Shapes match:", shap_left.shape == X.shape)

# Plot 1: Bar
plt.figure()
shap.summary_plot(shap_left, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("../output/shap_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved shap_bar.png")

# Plot 2: Beeswarm
plt.figure()
shap.summary_plot(shap_left, X, show=False)
plt.tight_layout()
plt.savefig("../output/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved shap_beeswarm.png")

# Plot 3: Dependence — overtime vs income interaction
ot_col = "OverTime_Binary" if "OverTime_Binary" in X.columns else "OverTime"
plt.figure()
shap.dependence_plot(ot_col, shap_left, X,
                     interaction_index="MonthlyIncome", show=False)
plt.tight_layout()
plt.savefig("../output/shap_dependence.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved shap_dependence.png")

# Plot 4: Waterfall for highest-risk employee
lgbm_proba = best_lgbm.predict_proba(X)[:, 1]
high_risk_idx = np.argmax(lgbm_proba)

base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(np.array(base_val).flat[-1])

shap_exp = shap.Explanation(
    values=shap_left[high_risk_idx],
    base_values=base_val,
    data=X.iloc[high_risk_idx].values,
    feature_names=X.columns.tolist()
)
plt.figure()
shap.waterfall_plot(shap_exp, show=False)
plt.tight_layout()
plt.savefig("../output/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.close()

print(f"\nHighest risk employee predicted probability: {lgbm_proba[high_risk_idx]:.1%}")
print(f"Actual outcome: {'LEFT' if y.iloc[high_risk_idx] == 1 else 'STAYED'}")
print("\nAll 4 SHAP plots saved!")
print("\nWeek 2 complete!")

Shapes match: True
Saved shap_bar.png
Saved shap_beeswarm.png
Saved shap_dependence.png

Highest risk employee predicted probability: 98.5%
Actual outcome: LEFT

All 4 SHAP plots saved!

Week 2 complete!


<Figure size 640x480 with 0 Axes>